# Tek Gorsel Analiz - YOLO / SAM2 / VLM

Bu notebook arayuz degildir. Once cekirdek analiz hattini Colab'da dogrulamak icin kullanilir:

- Resim upload edilir.
- YOLO v1/v2/v3 sonuclari uretilebilir.
- SAM2 sonucu, YOLO v3 kutularindan dogrudan maske iyilestirme olarak uretilebilir.
- Hibrit sonuc, filtrelenmis YOLO v3 + SAM2 olarak uretilebilir.
- VLM icin dort ayri rapor alinir: sadece resim, resim+YOLO, resim+SAM2, resim+hibrit.
- Her modelin cikti gorseli ve her VLM raporu Drive `reports/single_image_analysis` altina kaydedilir.


## 1. Repo ve Paketler

In [ ]:
from pathlib import Path
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt
!pip install -q gdown requests pillow

## 2. Drive ve Model Agirliklari

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
WEIGHTS_DOWNLOAD_DIR = Path('/content/shared_drive_weights')
if WEIGHTS_DOWNLOAD_DIR.exists():
    shutil.rmtree(WEIGHTS_DOWNLOAD_DIR)
WEIGHTS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

!gdown --folder '{WEIGHTS_DRIVE_FOLDER_URL}' -O /content/shared_drive_weights --remaining-ok

WEIGHT_FILE_IDS = {
    'elements-seg-v1-best.pt': '12GNVSipuIoK2BwVXx7UWMwpPNnh2Ap4A',
    'elements-seg-v2-aug-controlled-best.pt': '1vuSckiDRJxWLSWgNlWpEhJmX1mnPunM7',
    'elements-seg-v3-no-erasing-best.pt': '1OGtVBhNVBgJq1r0aQc5Eiu5wPz1Wd7ME',
}
if not list(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt')):
    print('No .pt found from folder download; trying direct file IDs...')
    for filename, file_id in WEIGHT_FILE_IDS.items():
        !gdown --id {file_id} -O /content/shared_drive_weights/{filename}

weights_dst = PROJECT_DIR / 'drive_weights'
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)
for pt in sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt')):
    shutil.copy2(pt, weights_dst / pt.name)

print('Weights copied to:', weights_dst)
for pt in sorted(weights_dst.rglob('*.pt')):
    print('-', pt)
print('Reports dir:', REPORTS_DIR)

## 3. SAM2 Kurulumu

Hibrit YOLO+SAM2 denemesi icin gerekli. Sadece YOLO denemek istersen bu hucreyi atlayabilirsin.

In [ ]:
%cd /content
!rm -rf /content/sam2
!git clone https://github.com/facebookresearch/sam2.git /content/sam2
%cd /content/sam2
!pip install -q -e .
!mkdir -p /content/sam2/checkpoints
!wget -q -nc -O /content/sam2/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
%cd /content/lejanter_doga_vlm_codex
print('SAM2 ready')

## 4. Resim Upload

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

UPLOAD_DIR = Path('/content/uploaded_single_image')
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
assert uploaded, 'Resim yuklenmedi.'
uploaded_name = next(iter(uploaded.keys()))
IMAGE_PATH = UPLOAD_DIR / uploaded_name
Path(uploaded_name).rename(IMAGE_PATH)
print('Image:', IMAGE_PATH)

## 5. Analizi Calistir

Ilk test icin VLM kapali. Once model ciktilarini dogrulayalim.

In [ ]:
RUN_VLM = False
VLM_MODEL = 'gpt-4o'

!cd /content/lejanter_doga_vlm_codex && python scripts/analyze_single_image.py \
  --image "$IMAGE_PATH" \
  --project-dir /content/lejanter_doga_vlm_codex \
  --weights-dir /content/lejanter_doga_vlm_codex/drive_weights \
  --reports-dir "$REPORTS_DIR" \
  --yolo-device 0 \
  --run-v1 \
  --run-v2 \
  --run-v3 \
  --run-sam2 \
  --run-hybrid \
  --sam2-dir /content/sam2 \
  --sam2-checkpoint /content/sam2/checkpoints/sam2.1_hiera_tiny.pt \
  --sam2-model-cfg configs/sam2.1/sam2.1_hiera_t.yaml \
  --sam2-device cuda

## 6. Son Ciktilari Goster

Bu hucre model cikti gorsellerini, JSON ozetini ve uretilen markdown raporunu gosterir.


In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage, Markdown
import json

analysis_root = REPORTS_DIR / 'single_image_analysis'
session_dir = sorted([p for p in analysis_root.iterdir() if p.is_dir()])[-1]
print('Session dir:', session_dir)

summary = json.loads((session_dir / 'summary.json').read_text())
display(Markdown('## Makine Ozeti'))
summary_text = json.dumps(summary, ensure_ascii=False, indent=2)[:9000]
display(Markdown(f'```json\n{summary_text}\n```'))

display(Markdown('## Model Cikti Gorselleri'))
for path in [summary.get('input_image')] + [item.get('visual') for item in summary.get('results', []) if item.get('visual')]:
    if path and Path(path).exists():
        print(path)
        display(IPImage(filename=path, width=900))

report_path = Path(summary.get('analysis_report', session_dir / 'analysis_report.md'))
if report_path.exists():
    display(Markdown('## Markdown Rapor'))
    display(Markdown(report_path.read_text()))


## 7. VLM Raporlarini Istersen Calistir

Bu hucre dort ayri VLM raporu uretir: sadece resim, YOLO destekli, SAM2 destekli ve hibrit destekli. OpenAI API key once Colab Secrets icinden `OPENAI_API_KEY`, `OPENAI_KEY` veya `OPENAI_API` adlariyla okunur. Bulunamazsa manuel giris ister.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    secret_key = userdata.get('OPENAI_API_KEY') or userdata.get('OPENAI_KEY') or userdata.get('OPENAI_API')
except Exception:
    secret_key = None

os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY') or secret_key or getpass('OpenAI API key: ')
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY bulunamadi.'
print('OpenAI API key hazir:', os.environ['OPENAI_API_KEY'][:7] + '...')

!cd /content/lejanter_doga_vlm_codex && python scripts/analyze_single_image.py \
  --image "$IMAGE_PATH" \
  --project-dir /content/lejanter_doga_vlm_codex \
  --weights-dir /content/lejanter_doga_vlm_codex/drive_weights \
  --reports-dir "$REPORTS_DIR" \
  --yolo-device 0 \
  --run-v3 \
  --run-sam2 \
  --run-hybrid \
  --sam2-dir /content/sam2 \
  --sam2-checkpoint /content/sam2/checkpoints/sam2.1_hiera_tiny.pt \
  --sam2-model-cfg configs/sam2.1/sam2.1_hiera_t.yaml \
  --sam2-device cuda \
  --run-vlm \
  --vlm-model gpt-4o


## 8. VLM Raporlarini Goster

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

analysis_root = REPORTS_DIR / 'single_image_analysis'
session_dir = sorted([p for p in analysis_root.iterdir() if p.is_dir()])[-1]
print('Session dir:', session_dir)
for report_path in sorted((session_dir / 'vlm_reports').glob('*.md')):
    display(Markdown(f'## {report_path.name}'))
    display(Markdown(report_path.read_text()))